In [1]:
#import
from torch import nn,optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from matplotlib import pyplot as plt
import os
import copy
import math
import sys
import importlib
from tqdm.auto import tqdm

DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'current device: {DEVICE}')

current device: cpu


c:\Users\darwin5991\.conda\envs\phh\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import main_function as mf
import pruning_function as pf
import quantization_function as qf
import memory_prep as mp
import layer_processing as lp

importlib.reload(mf)
importlib.reload(pf)
importlib.reload(qf)
importlib.reload(mp)
importlib.reload(lp)

<module 'layer_processing' from 'c:\\Users\\darwin5991\\Desktop\\programing\\CNN_NPU\\pytorch_practice\\layer_processing.py'>

In [3]:
path = '.'
model_dir, root, checkpoint_dir, output_dir, save_path, quant_save_path = mf.get_paths(path, print_paths=False)

model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True).to(DEVICE)

train_DL, val_DL, test_DL, test_DS= mf.load_data(root, batch_size=128)

CALL_SAVED=True
DO_TEST=False


Using cache found in C:\Users\darwin5991/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Files already downloaded and verified
Files already downloaded and verified


In [4]:
if DO_TEST or not CALL_SAVED:
    rcorrect,org_acc=mf.Test(model, test_DL, DEVICE, print_acc=True)

## pruning

In [5]:
if CALL_SAVED:
    finetuned_pruned_model = torch.load(save_path,weights_only=False,map_location=DEVICE)
    print("Loaded finetuned pruned model from saved file.")
else:
    file_path_torch = os.path.join(output_dir, 'psa_re_results_torch.pt')
    pruned_model, masks=pf.prune_model(model, org_acc, file_path_torch, DEVICE=DEVICE)
    pruned_rcorrect,pruned_acc=mf.Test(pruned_model, test_DL, DEVICE, print_acc=True)

    finetuned_pruned_model=pf.finetuning_pruned_model(pruned_model, train_DL, test_DL, masks, DEVICE)
    # finetuned_rcorrect,finetuned_acc=mf.Test(finetuned_pruned_model, test_DL, DEVICE, print_acc=True)

    torch.save(finetuned_pruned_model, save_path)
    print("Saved finetuned pruned model to file.")

if DO_TEST:
    finetuned_rcorrect,finetuned_acc=mf.Test(finetuned_pruned_model, test_DL, DEVICE, print_acc=True)

Loaded finetuned pruned model from saved file.


## quantization

In [6]:
if CALL_SAVED:
    input_sub, quanted_sub_layers, layer_params = qf.load_quanted_objects(quant_save_path, device=DEVICE)
else:
    input_sub, sub_layers= qf.separate_classifier_model(finetuned_pruned_model, num_layer=3, DEVICE=DEVICE)
    layer_params, quanted_sub_layers = qf.do_quantize(input_sub=input_sub, sub_layers=sub_layers, val_DL=val_DL, bit_width=8,  num_layers=3, DEVICE=DEVICE)
    qf.save_quanted_objects(input_sub, quanted_sub_layers, layer_params, quant_save_path)
    print("Saved quantized objects to file.")

if DO_TEST:
    qf.test_quantized(layer_params, quanted_sub_layers, input_sub, test_DL, num_layers=3, DEVICE=DEVICE)

Quantized objects loaded from: .\checkpoints\quanted_model.pth


In [7]:
# qf.print_layer_params_all(layer_params)
# qf.print_layer_params(layer_params,0)

## layer processing

In [17]:
importlib.reload(mf)
importlib.reload(pf)
importlib.reload(qf)
importlib.reload(mp)
importlib.reload(lp)

<module 'layer_processing' from 'c:\\Users\\darwin5991\\Desktop\\programing\\CNN_NPU\\pytorch_practice\\layer_processing.py'>

In [7]:
# memory 파일 저장 경로
sim_dir = os.path.join(path, "sim_file")
os.makedirs(sim_dir, exist_ok=True)

# 샘플 1개 준비 (file for memory 용)
test_iter = iter(test_DS)
images, labels = next(test_iter)
images = images.unsqueeze(0).to(DEVICE)

In [21]:
raw1, pe_static_debug1, pe_stage_debug1, encoded_data1 = lp.process_layer_to_memory(
    images=images,
    input_sub=input_sub,                 # 첫 레이어는 input_sub 사용
    layer_idx=0,
    layer_params=layer_params,
    quanted_sub_layers=quanted_sub_layers,
    sim_dir=sim_dir,
    device=DEVICE,
    split_size=128,
    bit_width=8,
)
print(f" Scale_factor : {pe_static_debug1['Scale_factor']}")
dbg = lp.Dbg(pe_static_debug1, pe_stage_debug1)

dbg.n()

 Scale_factor : 135.0
[933, 907, 891, 897]
[827, 831, 786, 830]
[831, 847, 831, 873]
[724, 713, 670, 716]


In [23]:
# print(encoded_data1.keys())
# print(len(encoded_data1['input_encoded_hex'][0]))
# print(len(encoded_data1['qw_data_mem_hex'][0]))

# print(pe_static_debug1.keys())
# print(pe_static_debug1['PE0'].keys())
# print(pe_static_debug1['PE0']['input'].shape)
# print(pe_static_debug1['PE0']['output'].shape)
# print(pe_static_debug1['PE0']['weight_splits'][0].shape)
# print(pe_static_debug1['PE0']['bias'].shape)
# print(pe_static_debug1['PE0']['weight_csr'][0].keys())

# print(pe_stage_debug1.keys())
# print(pe_stage_debug1['PE0'].keys())

In [28]:
dbg.s10(pe=0, key="input", idx=3)
print()
dbg.s10(pe=1, key="bias", idx=0)


PE0.input[30:40]
tensor([ -70., -126., -104., -128., -114., -118., -128., -128., -117., -119.])

PE1.bias[0:10]
tensor([251380.,      0.,      0., 237061., 221761., 255370., 206376., 209568.,
             0., 260152.])


In [ ]:
# stage  :  mac_stage0, mac_stage1, mac_stage2, mac_stage3,
#           bias_added, relu_out, requant_out, clamped_out
dbg.p10(pe=2, stage="mac_stage2", idx=4)


PE2.mac_stage2[40:50]
tensor([      0.,       0., -133726., -128075., -113303., -176585.,       0.,
        -176480.,       0., -119282.])


In [34]:
dbg.m(pe=0, stage=1, row=0)


PE0, stage1, row0
input_split=1, row_num=13
[0] col=4, val=21, in=-75, mul=-1575, acc=-1575
[1] col=20, val=40, in=-118, mul=-4720, acc=-6295
[2] col=24, val=24, in=-104, mul=-2496, acc=-8791
[3] col=30, val=32, in=-68, mul=-2176, acc=-10967
[4] col=67, val=32, in=-127, mul=-4064, acc=-15031
[5] col=69, val=27, in=-113, mul=-3051, acc=-18082
[6] col=75, val=25, in=-108, mul=-2700, acc=-20782
[7] col=83, val=34, in=-128, mul=-4352, acc=-25134
[8] col=89, val=36, in=-115, mul=-4140, acc=-29274
[9] col=93, val=38, in=-116, mul=-4408, acc=-33682
[10] col=113, val=-21, in=-128, mul=2688, acc=-30994
[11] col=118, val=25, in=-88, mul=-2200, acc=-33194
[12] col=126, val=50, in=-115, mul=-5750, acc=-38944


-38944